# Lab 3: Transformers y HuggingFace en Español
**Curso de Maestría en NLP - Semana 1 - Sábado 11:30 AM (Módulo 4)**

| | |
|---|---|
| **Sesión** | Semana 1, Sábado - Módulo 4 |
| **Entorno** | Google Colab gratuito (CPU es suficiente para inferencia con modelos pequeños) |

### Qué vas a aprender
- Por qué los embeddings **estáticos** (Word2Vec/GloVe) no capturan el contexto
- Qué son los embeddings **contextualizados** que producen los Transformers
- Cómo usar **HuggingFace pipelines** para resolver tareas de NLP en español en 3 líneas
- Análisis de sentimiento y Reconocimiento de Entidades (NER) sin entrenar ningún modelo
- Cómo comparar un Transformer con el clasificador clásico (BoW/TF-IDF) del Lab 3


## Configuración del entorno


In [ ]:
# @title Configuración - instalar librerías
# transformers + torch para los pipelines; gensim para los embeddings estáticos de GloVe.
!pip install -q transformers torch gensim

import warnings
warnings.filterwarnings('ignore')
import numpy as np
print('Entorno listo.')


---
## Paso 1: El Problema de la Ambigüedad — Motivación

En el Lab 1 y el Lab 2 representamos el texto con **Bag of Words** y **TF-IDF**: cada palabra
es una posición fija en un vector, sin ningún conocimiento de significado. Word2Vec y GloVe
dieron un paso adelante: aprenden un **vector por palabra** que captura similitud semántica
(`rey - hombre + mujer ≈ reina`).

#### El problema: los embeddings estáticos dan UN solo vector por palabra
La palabra **"banco"** significa cosas distintas según el contexto:
- *"Fui al **banco** a depositar"* → institución financiera
- *"Me senté en el **banco** del parque"* → asiento

Con GloVe, ambos usos comparten **exactamente el mismo vector**. El modelo no puede
distinguir los dos sentidos, porque el vector se calculó una sola vez, sin contexto.

#### La solución: embeddings contextualizados
Los **Transformers** (BERT, RoBERTa, etc.) producen un vector **distinto para cada aparición**
de la palabra, en función de las palabras que la rodean. Ese es el salto conceptual de este lab.


In [ ]:
# @title Paso 1: Demo con GloVe — un solo vector por palabra (sin contexto)
import gensim.downloader as api

# GloVe entrenado en Wikipedia + Gigaword (100 dimensiones). En inglés porque los
# vectores GloVe pre-entrenados más comunes son en inglés; el concepto es idéntico
# para español. La palabra ambigua 'bank' = banco (financiero) / orilla / banca.
print('Descargando GloVe (glove-wiki-gigaword-100)... puede tardar la primera vez.')
modelo = api.load('glove-wiki-gigaword-100')
print('GloVe cargado. Vocabulario:', len(modelo.index_to_key), 'palabras')

# El vector de 'bank' es UNO solo, sin importar el contexto:
vec_bank = modelo['bank']
print('\nShape del vector de "bank":', vec_bank.shape)
print('Primeras 5 dimensiones:', np.round(vec_bank[:5], 3))

# Simulamos dos oraciones con 'banco' en sentidos distintos.
# Como GloVe es estático, el vector de 'bank' es identico en ambas:
vec_bank_oracion1 = modelo['bank']  # 'Fui al banco a depositar'
vec_bank_oracion2 = modelo['bank']  # 'Me sente en el banco del parque'
iguales = np.array_equal(vec_bank_oracion1, vec_bank_oracion2)
print(f'\n¿El vector de "bank" es identico en ambas oraciones? {iguales}')


In [ ]:
# @title Paso 1: La similitud NO cambia con el contexto (esa es la limitación)
# Comparamos 'bank' con palabras del sentido financiero vs. del sentido 'orilla/parque'.
financieras = ['money', 'credit', 'loan']
lugar       = ['river', 'park', 'bench']

print('Similitud coseno de "bank" (vector único de GloVe):')
print('  Sentido FINANCIERO:')
for w in financieras:
    if w in modelo:
        print(f'    bank ~ {w:8s}: {modelo.similarity("bank", w):.3f}')
print('  Sentido LUGAR/ASIENTO:')
for w in lugar:
    if w in modelo:
        print(f'    bank ~ {w:8s}: {modelo.similarity("bank", w):.3f}')

print('\nClave: da igual en qué oración aparezca "banco" — GloVe usa SIEMPRE el mismo')
print('vector, así que estas similitudes nunca cambian. No hay desambiguación por contexto.')
print('Los Transformers resuelven justo esto: un vector distinto por cada contexto.')


### Reflexión 1 - Embeddings estáticos vs. contextualizados

> 1. ¿Por qué un único vector por palabra es un problema para palabras polisémicas como "banco", "gato" o "vela"?
> 2. ¿Qué información necesita un modelo para asignar el sentido correcto a "banco"?
> 3. Word2Vec/GloVe resuelven analogías (`rey-hombre+mujer≈reina`). ¿Por qué eso no basta para la ambigüedad de sentido?

# === PAUSA: DISCUSIÓN ===


---
## Paso 2: HuggingFace Pipelines — Inferencia en 3 Líneas

#### ¿Qué es HuggingFace?
**HuggingFace** es, en la práctica, el "GitHub de los modelos de IA": un repositorio público
con cientos de miles de modelos pre-entrenados (muchos en español) que puedes descargar y usar
gratis. En lugar de entrenar un Transformer desde cero (que cuesta miles de dólares en GPU),
reutilizamos uno que ya aprendió el idioma.

#### ¿Qué es un pipeline?
Un **pipeline** de la librería `transformers` encapsula los 3 pasos de la inferencia en una sola
llamada: (1) tokeniza el texto, (2) lo pasa por el modelo Transformer, (3) convierte la salida
en un resultado legible (etiqueta + score). Esto nos deja resolver una tarea en **3 líneas**.

#### ¿Por qué podemos usar modelos en español sin entrenar nada?
Porque otros equipos ya entrenaron (fine-tuning) modelos como BETO o RoBERTa-BNE sobre datos en
español para tareas concretas (sentimiento, NER) y los publicaron. Nosotros solo hacemos
**inferencia**: aprovechamos ese conocimiento sin datos ni GPU propios.


### 2a: Análisis de Sentimiento en Español

> **Nota sobre el modelo:** el enunciado original sugería
> `PlanTL-GOB-ES/roberta-base-bne-cap-es-menéame`, pero ese repositorio **no existe** en
> HuggingFace (id inválido, con acento). Usamos en su lugar
> **`pysentimiento/robertuito-sentiment-analysis`**, un modelo RoBERTa entrenado sobre
> **tweets en español** — ideal porque los datos del Lab 3 también eran tweets. Devuelve las
> etiquetas `POS`, `NEU`, `NEG`. (Alternativa válida: `finiteautomata/beto-sentiment-analysis`.)


In [ ]:
# @title Paso 2a: Sentimiento en español con un pipeline de HuggingFace
from transformers import pipeline

# 3 líneas: cargamos el pipeline con un modelo de sentimiento en español (tweets).
clf = pipeline('sentiment-analysis',
               model='pysentimiento/robertuito-sentiment-analysis')

# 10 tweets de ejemplo (mismo estilo que el dataset de Twitter del Lab 3).
tweets = [
    'que pesimo servicio, llevo horas esperando y nadie responde',
    'me encanta esta marca, la calidad es increible',
    'alguien sabe a que hora abren hoy?',
    'una decepcion total, no vuelvo a comprar nunca mas',
    'gracias por el envio tan rapido, llego antes de lo esperado',
    'acabo de recibir el pedido, aun no lo he abierto',
    'el mejor servicio al cliente que he tenido, felicitaciones',
    'estoy harto de que siempre fallen cuando mas los necesito',
    'segun el comunicado cambiaran los horarios el proximo mes',
    'super recomendado, supero todas mis expectativas',
]

# El modelo devuelve etiquetas POS/NEU/NEG; las mapeamos a las clases del Lab 3.
MAP_SENT = {'NEG': 'negativo', 'NEU': 'neutro', 'POS': 'positivo'}

print('SENTIMIENTO PREDICHO POR EL TRANSFORMER')
print('=' * 70)
for tw in tweets:
    r = clf(tw)[0]
    etiqueta = MAP_SENT.get(r['label'], r['label'])
    print(f'  [{etiqueta:9s} {r["score"]:.2f}]  {tw[:60]}')


#### Comparación con el Lab 3 (BoW/TF-IDF + LogReg)

En el Lab 3, el mejor modelo clásico (TF-IDF + Regresión Logística) alcanzó ~0.54 de F1 macro
sobre los tweets de sentimiento. El Transformer que acabas de usar:

- **No requiere entrenamiento** ni preprocesamiento (ni stemming ni quitar stopwords).
- Entiende **contexto** y negaciones ("no lo recomiendo") que BoW trata como palabras sueltas.
- Suele lograr bastante más exactitud en textos difíciles (ironía, negación, jerga).

**¿Cuándo importa la diferencia?** Cuando el texto es corto, informal y depende del orden y la
negación (redes sociales). **¿Cuándo NO tanto?** Para clasificación temática con vocabulario muy
marcado (deporte vs. finanzas), donde BoW/TF-IDF ya es fuerte, más rápido y más interpretable.


### Reflexión 2a - Transformer vs. modelo clásico

> 1. ¿En qué tweets crees que el Transformer acierta y BoW/TF-IDF fallaría (pista: negación, ironía)?
> 2. El Transformer es más lento y menos interpretable. ¿En qué escenario preferirías igualmente TF-IDF + LogReg?
> 3. ¿Qué riesgo hay en confiar en un modelo pre-entrenado sin validarlo con TUS datos?

# === PAUSA: DISCUSIÓN ===


### 2b: Reconocimiento de Entidades (NER) en Español

El **NER (Named Entity Recognition)** detecta y clasifica entidades dentro del texto:
**PER** (personas), **ORG** (organizaciones), **LOC** (lugares) y **MISC** (misceláneas).
Es clave para extraer información estructurada de noticias, contratos o reportes.

Usamos **`mrm8488/bert-spanish-cased-finetuned-ner`** (BETO afinado para NER en español).
El parámetro `aggregation_strategy='simple'` agrupa los sub-tokens en entidades completas
(p. ej. une "Banco" + "##lombia" en la entidad `Bancolombia`).


In [ ]:
# @title Paso 2b: NER en español con un pipeline de HuggingFace
ner = pipeline('ner',
               model='mrm8488/bert-spanish-cased-finetuned-ner',
               aggregation_strategy='simple')

# 5 noticias financieras en español.
noticias = [
    'Bancolombia anuncio nuevas inversiones en Medellin durante el ultimo trimestre.',
    'El presidente de Ecopetrol se reunio con inversionistas en Bogota y Nueva York.',
    'La Bolsa de Valores de Colombia cerro al alza tras el reporte de Grupo Nutresa.',
    'Avianca firmo un acuerdo con Airbus para renovar su flota en America Latina.',
    'El Banco de la Republica, dirigido por Leonardo Villar, mantuvo la tasa de interes.',
]

print('ENTIDADES DETECTADAS (NER)')
print('=' * 70)
for texto in noticias:
    print(f'\nNoticia: {texto}')
    entidades = ner(texto)
    if not entidades:
        print('  (sin entidades detectadas)')
    for e in entidades:
        grupo = e.get('entity_group', e.get('entity', '?'))
        print(f'  - {e["word"]:25s} -> {grupo:5s} (score {e["score"]:.2f})')


In [ ]:
# @title Paso 2b (visual): conteo de entidades por tipo
import matplotlib.pyplot as plt
from collections import Counter

conteo_tipos = Counter()
for texto in noticias:
    for e in ner(texto):
        grupo = e.get('entity_group', e.get('entity', 'MISC'))
        conteo_tipos[grupo] += 1

if conteo_tipos:
    tipos   = list(conteo_tipos.keys())
    valores = [conteo_tipos[t] for t in tipos]
    colores = ['#1B3A6B', '#0EA5A0', '#16A34A', '#E8821A', '#7F8C8D']
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(tipos, valores, color=colores[:len(tipos)], alpha=0.85)
    for i, v in enumerate(valores):
        ax.text(i, v + 0.05, str(v), ha='center', fontweight='bold')
    ax.set_ylabel('Número de entidades')
    ax.set_title('Entidades detectadas por tipo (5 noticias)', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('No se detectaron entidades.')

print('Tipos: PER=persona, ORG=organizacion, LOC=lugar, MISC=miscelanea')


### Reflexión 2b - NER

> 1. ¿Qué entidades detectó bien y cuáles confundió o partió mal? ¿Por qué es difícil el caso de "Banco de la República"?
> 2. ¿Cómo usarías NER en un producto real (p. ej. leer contratos, monitorear noticias de la competencia)?
> 3. ¿Por qué un enfoque de Bag of Words no serviría para esta tarea?

# === PAUSA: DISCUSIÓN ===


---
## Paso 3: Embeddings Contextualizados con BETO

En el Paso 1 vimos que GloVe le da a "banco" **un único vector**, sin importar el contexto.
Ahora cerramos el círculo: **BETO** (BERT entrenado en español, `dccuchile/bert-base-spanish-wwm-cased`)
produce un vector **distinto para cada aparición** de la palabra, según la oración completa.

Vamos a tomar la MISMA palabra "banco" en dos oraciones con sentidos diferentes y comparar sus
vectores. Si el modelo es realmente contextual, los dos vectores **no** serán idénticos, y cada
uno se parecerá más a las palabras de su propio contexto.


In [ ]:
# @title Paso 3: BETO da un vector distinto a 'banco' según el contexto
import torch
from transformers import AutoTokenizer, AutoModel

BETO = 'dccuchile/bert-base-spanish-wwm-cased'
tokenizer = AutoTokenizer.from_pretrained(BETO)
modelo_beto = AutoModel.from_pretrained(BETO)
modelo_beto.eval()

def embedding_de_palabra(oracion, palabra='banco'):
    """Devuelve el vector contextual de `palabra` dentro de `oracion`."""
    enc = tokenizer(oracion, return_tensors='pt')
    with torch.no_grad():
        salida = modelo_beto(**enc).last_hidden_state[0]  # (n_tokens, 768)
    tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    # Buscar el primer sub-token que contenga la palabra objetivo
    idx = next(i for i, t in enumerate(tokens) if palabra in t.lower())
    return salida[idx]

oracion_fin    = 'Fui al banco a depositar dinero'
oracion_parque = 'Me sente en el banco del parque'

v_fin    = embedding_de_palabra(oracion_fin)
v_parque = embedding_de_palabra(oracion_parque)

cos = torch.nn.functional.cosine_similarity(v_fin, v_parque, dim=0).item()
print(f'Dimension del vector contextual de BETO: {v_fin.shape[0]}')
print(f'\nOracion 1 (financiero): "{oracion_fin}"')
print(f'Oracion 2 (parque):     "{oracion_parque}"')
print(f'\nSimilitud coseno entre los dos "banco": {cos:.3f}')
print('=> Es < 1.0: BETO asigna vectores DISTINTOS a la misma palabra segun el contexto.')
print('   (Con GloVe, del Paso 1, esta similitud seria exactamente 1.000.)')


In [ ]:
# @title Paso 3 (comprobación): cada 'banco' se parece más a su propio contexto
# Comparamos cada vector contextual de 'banco' con el embedding contextual de una
# palabra clave de cada sentido: 'dinero' (financiero) y 'parque' (lugar).
v_dinero = embedding_de_palabra('Fui al banco a depositar dinero', palabra='dinero')
v_arbol  = embedding_de_palabra('Me sente en el banco del parque',  palabra='parque')

def cos(a, b):
    return torch.nn.functional.cosine_similarity(a, b, dim=0).item()

print('Similitud del "banco" de cada oracion con la palabra clave de su contexto:')
print(f'  banco(financiero) ~ dinero : {cos(v_fin, v_dinero):.3f}')
print(f'  banco(parque)     ~ parque : {cos(v_parque, v_arbol):.3f}')
print()
print('A diferencia de GloVe, aqui el vector de "banco" SI cambia con la oracion,')
print('acercandose al significado correcto en cada caso. Eso es un embedding contextual.')


### Reflexión 3 - Embeddings contextualizados

> 1. ¿Por qué la similitud entre los dos "banco" de BETO es < 1.0 mientras que con GloVe sería exactamente 1.0?
> 2. ¿De dónde saca BETO la información para diferenciar los dos sentidos si la palabra escrita es idéntica?
> 3. ¿Cómo aprovecharías estos vectores contextuales como *features* para un clasificador (en lugar de BoW/TF-IDF)?

# === PAUSA: DISCUSIÓN ===


---
## Paso 4: Fine-Tuning Conceptual con LoRA

Hasta ahora solo hicimos **inferencia** con modelos que otros entrenaron. ¿Qué pasa si queremos
**adaptar** un Transformer a NUESTROS datos (p. ej. tweets de nuestra propia marca con jerga
particular)? Eso es **fine-tuning**. Este paso es **conceptual**: explicamos el porqué y mostramos
el esqueleto de código, sin ejecutar un entrenamiento (que requeriría GPU y tiempo).

#### El problema del fine-tuning completo
Un modelo como BETO tiene ~110 millones de parámetros. Afinar **todos** requiere mucha memoria de
GPU, guardar una copia completa del modelo por cada tarea, y es lento y caro.

#### La solución: LoRA (Low-Rank Adaptation)
**LoRA** congela los pesos originales del modelo y solo entrena unas pocas **matrices pequeñas de
bajo rango** que se "suman" a las capas de atención. En la práctica:
- Se entrena **<1%** de los parámetros (a veces 0.1%).
- El adaptador resultante pesa unos pocos MB (vs. cientos de MB del modelo completo).
- Puedes tener **muchos adaptadores** (uno por tarea/cliente) sobre un mismo modelo base.
- La calidad es comparable al fine-tuning completo en muchas tareas.

Es una técnica de **PEFT** (Parameter-Efficient Fine-Tuning), disponible en la librería `peft` de HuggingFace.


In [ ]:
# @title Paso 4: Esqueleto CONCEPTUAL de fine-tuning con LoRA (no se ejecuta el entrenamiento)
# Esta celda es ilustrativa. Para correrla de verdad necesitas: pip install peft datasets
# accelerate, un dataset etiquetado y (idealmente) una GPU. En Colab, Runtime -> GPU.
#
# from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
#                           TrainingArguments, Trainer)
# from peft import LoraConfig, get_peft_model, TaskType
#
# BASE = 'dccuchile/bert-base-spanish-wwm-cased'
# tokenizer = AutoTokenizer.from_pretrained(BASE)
# modelo = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=3)
#
# # 1) Configurar LoRA: solo se entrenan matrices de bajo rango (r=8) en la atencion.
# lora_config = LoraConfig(
#     task_type=TaskType.SEQ_CLS,   # clasificacion de secuencias (sentimiento 3 clases)
#     r=8,                          # rango de las matrices LoRA (pequeño)
#     lora_alpha=16,                # factor de escala
#     lora_dropout=0.1,
#     target_modules=['query', 'value'],  # capas de atencion a adaptar
# )
# modelo_lora = get_peft_model(modelo, lora_config)
#
# # 2) Ver cuantos parametros se entrenan realmente (deberia ser <1%).
# modelo_lora.print_trainable_parameters()
#
# # 3) Entrenar como siempre (Trainer), pero solo se actualizan los pesos LoRA.
# #    args = TrainingArguments(output_dir='./lora-sentimiento', num_train_epochs=3, ...)
# #    trainer = Trainer(model=modelo_lora, args=args, train_dataset=..., eval_dataset=...)
# #    trainer.train()
#
# # 4) Guardar SOLO el adaptador (pocos MB), no el modelo completo.
# #    modelo_lora.save_pretrained('./adaptador-lora-sentimiento')

print('Paso 4 es conceptual: arriba esta el esqueleto de LoRA comentado.')
print('Idea clave: congelar el modelo base y entrenar <1% de parametros (matrices de bajo rango).')
print('Resultado: adaptadores de pocos MB, un modelo base compartido, entrenamiento barato.')


### Reflexión 4 - Fine-tuning y LoRA

> 1. ¿Por qué entrenar <1% de los parámetros puede dar resultados comparables a afinar el 100%?
> 2. Tienes 5 clientes, cada uno con su jerga. ¿Por qué LoRA es mejor que guardar 5 modelos completos?
> 3. ¿Cuándo NO vale la pena hacer fine-tuning y basta con un pipeline pre-entrenado (como en el Paso 2)?

# === PAUSA: DISCUSIÓN ===


---
## Reflexiones Finales

# === PAUSA: DISCUSIÓN FINAL ===

> **1. Estático vs. contextual:** con tus palabras, ¿cuál es la diferencia entre el vector de
> GloVe para "banco" y el vector que produciría BERT para "banco" en dos oraciones distintas?
>
> **2. Pipelines:** ¿qué tres pasos encapsula un pipeline de HuggingFace? ¿Por qué eso acelera
> tanto el desarrollo?
>
> **3. Cuándo usar qué:** para un cliente con 500 tweets/día y necesidad de interpretar por qué
> se clasifica algo, ¿usarías el Transformer, el modelo del Lab 3, o una combinación?
>
> **4. Siguiente paso:** hoy solo hicimos *inferencia*. ¿Qué haría falta para *fine-tunear* uno de
> estos modelos con TUS propios datos etiquetados?

---

## Resumen del Lab

| Concepto | Aprendizaje clave |
|---|---|
| **Embedding estático** | Un vector fijo por palabra (Word2Vec/GloVe): no distingue sentidos |
| **Embedding contextual** | Un vector por aparición (Transformers): resuelve la ambigüedad |
| **HuggingFace** | Repositorio público de modelos pre-entrenados ("GitHub de la IA") |
| **pipeline()** | Tokeniza + infiere + formatea en una sola llamada |
| **Sentiment analysis** | Clasificación de sentimiento sin entrenar (supera a BoW en texto difícil) |
| **NER** | Extrae entidades (PER/ORG/LOC/MISC): imposible con Bag of Words |
